# [LAB12] 딥러닝 > 신경망의 이해 > 04. CheckPoint 활용

## 📘 #01. 준비작업

### 📝 [1] 패키지 가져오기

In [ ]:
import os
import numpy as np
from hossam import *
from pandas import concat, DataFrame
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from tqdm.keras import TqdmCallback

## 📘 #02. 저장된 CheckPoint 로딩

### 📝 [1] 대상 파일의 경로 생성

In [ ]:
save_dir = os.path.join(os.getcwd(), "tensorflow_checkpoint")
check_points = os.listdir(save_dir)
last_point = sorted(check_points)[-1]
last_point_path = os.path.join(save_dir, last_point)
last_point_path

### 📝 [2] 선정된 파일을 모델로 로드함

## 📘 #03. 모델 활용

### 📝 [1] 추가 학습

#### ✏ 데이터 불러오기 (원래는 모델이 학습한 적 없는 새로운 데이터로 수행해야 하지만 여기서는 기존 데이터 재사용)

In [ ]:
model = load_model(last_point_path)
model.summary()

#### ✏ 훈련/검증 데이터 분리

In [ ]:
origin = load_data("logical_xor")
df = concat([origin] * 10, ignore_index=True)
df.describe()

In [ ]:
yname = "target"
x = df.drop(columns=[yname])
y = df[yname]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
cwd = os.getcwd()
target_dir = os.path.join(cwd, "tensorflow_checkpoint")
if not os.path.exists(target_dir):
    os.mkdir(target_dir)
checkpoint_path = os.path.join(target_dir, "model04-cp-{epoch:04d}-ckpt.keras")
checkpoint_path

#### ✏ 추가학습 (이전 실습 코드 재사용)

모델이 추가 데이터로 학습을 수행하고, 그 과정(혹은 최종 모델)을 체크포인트로 저장한다.

### 📝 [2] 로드한 모델을 통해 예측치 얻기

반드시 추가 학습을 한 후에 사용해야 하는것은 아니다. 분석과정에서 생성된 체크포인트를 백엔드 등의 다른 시스템이 로드하는 것도 가능하다.

In [ ]:
%%time
result = model.fit(
    x_train, y_train,
    epochs=500,
    validation_data=(x_test, y_test),
    verbose=0,
    callbacks=[
        TqdmCallback(verbose=1),
        ModelCheckpoint(filepath=checkpoint_path, monitor='val_loss', verbose=0, save_best_only=True),
        EarlyStopping(monitor='val_loss', patience=5, min_delta=0.0001),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0, verbose=1)
    ]
)
result

In [ ]:
r = model.predict(x_test, verbose=0)
r = np.where(r > 0.5, 1, 0)
r